# Connections: Global Partition Prediction

Author: Will Blanton

Experimental copy of `connections-solver.ipynb`. Predict one partition of all 16 words into four disjoint groups of four. Reuses the original data cleaning, embeddings, and attention components. The original notebook and models are unchanged.

## Imports and configuration

In [1]:
import ast
import itertools
import random
import sys
from datetime import datetime
from pathlib import Path

import numpy as np
import optuna
import pandas as pd
import torch
from sklearn.model_selection import KFold
from sentence_transformers import SentenceTransformer
from torch.utils.data import Dataset, DataLoader

sys.path.insert(0, str(Path("src").resolve()))
from nyt_connections_solver.partition import (
    PartitionTransformer, partition_loss, decode_partition, evaluate_partitions,
)
from nyt_connections_solver.partition_features import PartitionFeatureEncoder

DATASET_PATH = Path("data/connections.csv")
EMBEDDING_MODEL = "all-mpnet-base-v2"
SEED = 0
BATCH_SIZE = 16
EPOCHS = 200  # Maximum; validation early stopping usually ends training sooner.
EARLY_STOPPING_PATIENCE = 30
EARLY_STOPPING_MIN_DELTA = 1e-4  # Minimum validation CE improvement.
RUN_OPTUNA = True
OPTUNA_TRIALS = 12
N_SPLITS = 5
# Reuse the existing experiment's exact dates for comparison.
# Set to None to create the original seeded 80/10/10 date split.
SPLIT_CHECKPOINT = Path("weights/set_transformer_optuna_mixed_20261008_220606.pt")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def seed_experiment(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_experiment(SEED)
print("Training and evaluation device:", device)


Training and evaluation device: cuda


## Dataset and puzzle-level splits

Keep all words from a date in the same split. The two data repairs and April 1 exclusions below are copied from the ranking notebook. No category names are used as features.

In [2]:
connections_df = pd.read_csv(DATASET_PATH, index_col=0)
# connections_df.drop(columns='category', inplace=True)

# fix issues with incorrect format... (not worth going in and adding a mechanism to correct...)
connections_df.loc[1298, "connections"] = "['line', 'plane', 'point', 'solid']"
connections_df.loc[1892, "connections"] = "['abyss', 'fly', 'matrix', 'thing']"

# remove april fools samples since they include emojis or other potentially noisy samples
connections_df = connections_df[~connections_df['date'].str.contains("04-01")]
connections_df.reset_index(drop=True, inplace=True)

connections_df['connections'] = connections_df['connections'].apply(ast.literal_eval)
connections_df.head()

,date,category,connections
0,2023-06-12,palindromes,"[kayak, level, mom, race car]"
1,2023-06-12,keyboard keys,"[option, return, shift, tab]"
2,2023-06-12,nba teams,"[bucks, heat, jazz, nets]"
3,2023-06-12,wet weather,"[hail, rain, sleet, snow]"
4,2023-06-13,letter homophones,"[are, queue, sea, why]"


In [3]:
if SPLIT_CHECKPOINT is not None:
    split_checkpoint = torch.load(SPLIT_CHECKPOINT, map_location="cpu", weights_only=True)
    split_dates = split_checkpoint["split_dates"]
    train_dates = np.asarray(split_dates["train"])
    val_dates = np.asarray(split_dates["val"])
    test_dates = np.asarray(split_dates["test"])
    del split_checkpoint
else:
    seed_experiment(SEED)
    # split data
    dates = connections_df["date"].unique()
    train_idx = random.sample(range(len(dates)), k=int(len(dates) * .8))
    train_dates = dates[train_idx]
    test_dates = dates[~np.isin(range(len(dates)), train_idx)]
    test_dates = np.random.permutation(test_dates)
    split_idx = len(test_dates) // 2
    val_dates = test_dates[split_idx:]
    test_dates = test_dates[:split_idx]
    print(f"Total Length: {len(dates)}")
    print(f"Train length: {len(train_dates)}")
    print(f"Val length: {len(val_dates)}")
    print(f"Test length: {len(test_dates)}")

available_dates = set(connections_df["date"])
split_sets = [set(dates) for dates in (train_dates, val_dates, test_dates)]
if any(len(dates) != len(set(dates)) for dates in (train_dates, val_dates, test_dates)):
    raise ValueError("Split dates must be unique")
if any(split_sets[a] & split_sets[b] for a, b in ((0, 1), (0, 2), (1, 2))):
    raise ValueError("Puzzle splits overlap")
if not set.union(*split_sets) <= available_dates or not all(split_sets):
    raise ValueError("Dataset is missing checkpoint dates or contains an empty split")
print({name: len(dates) for name, dates in
       (("train", train_dates), ("val", val_dates), ("test", test_dates))})


{'train': 968, 'val': 121, 'test': 121}


## Word features

Semantic embeddings use the same frozen pretrained model as the ranking experiment. Structural statistics, Soundex vectors, and character n-gram SVD are included. Character vocabulary/SVD and structural normalization are fitted separately inside each training fold, so held-out fold words do not shape those transforms.

In [4]:
all_words = list(dict.fromkeys(itertools.chain.from_iterable(connections_df["connections"])))
word_embedder = SentenceTransformer(EMBEDDING_MODEL, device=str(device))
with torch.inference_mode():
    word_vectors = word_embedder.encode(
        all_words, convert_to_tensor=True, show_progress_bar=True,
    ).cpu()
D = word_vectors.shape[1]
word_to_row = {word: i for i, word in enumerate(all_words)}
semantic_vectors = {word: word_vectors[i].numpy() for word, i in word_to_row.items()}
# Features are cached on CPU. Free the encoder's GPU memory before training.
word_embedder.to("cpu")
del word_embedder
if device.type == "cuda":
    torch.cuda.empty_cache()
print("Unique words:", len(all_words), "Feature dimension:", D)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/274 [00:00<?, ?it/s]

Unique words: 8746 Feature dimension: 768


In [5]:
def words_for_dates(dates):
    grouped = connections_df.loc[connections_df["date"].isin(dates)].groupby("date")["connections"].agg(list)
    return list(itertools.chain.from_iterable(
        itertools.chain.from_iterable(grouped.loc[date]) for date in dates
    ))

class PartitionDataset(Dataset):
    def __init__(self, puzzle_df, dates, feature_bank, shuffle_words=False):
        self.shuffle_words = shuffle_words
        self.examples = []
        grouped = puzzle_df.groupby("date")["connections"].agg(list)
        for date in dates:
            groups = grouped.loc[date]
            if len(groups) != 4 or any(len(group) != 4 for group in groups):
                raise ValueError(f"Expected four groups of four on {date}")
            words = list(itertools.chain.from_iterable(groups))
            if len(set(words)) != 16 or any(not isinstance(w, str) for w in words):
                raise ValueError(f"Expected 16 distinct words on {date}")
            self.examples.append({
                "x": {name: values[[word_to_row[word] for word in words]]
                       for name, values in feature_bank.items()},
                "targets": torch.arange(4).repeat_interleave(4),  # (16,)
                "date": str(date),
            })

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, index):
        example = self.examples[index]
        order = torch.randperm(16) if self.shuffle_words else torch.arange(16)
        return {
            "x": {name: values[order] for name, values in example["x"].items()},
            "targets": example["targets"][order],
            "date": example["date"],
        }

def build_feature_bank(fit_dates):
    encoder = PartitionFeatureEncoder(semantic_vectors).fit(words_for_dates(fit_dates))
    arrays = encoder.encode(all_words)
    bank = {name: torch.from_numpy(values) for name, values in arrays.items()}
    return encoder, bank

def make_loader(dates, feature_bank, training=False):
    dataset = PartitionDataset(connections_df, dates, feature_bank, shuffle_words=training)
    return DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=training)

fold_splits = KFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
cv_loaders = []
for train_idx, heldout_idx in fold_splits.split(train_dates):
    fold_train_dates, fold_val_dates = train_dates[train_idx], train_dates[heldout_idx]
    _, fold_bank = build_feature_bank(fold_train_dates)
    cv_loaders.append((make_loader(fold_train_dates, fold_bank, training=True),
                       make_loader(fold_val_dates, fold_bank)))

final_feature_encoder, final_feature_bank = build_feature_bank(train_dates)
FEATURE_DIMS = final_feature_encoder.feature_dims
train_loader = make_loader(train_dates, final_feature_bank, training=True)
val_loader = make_loader(val_dates, final_feature_bank)
test_loader = make_loader(test_dates, final_feature_bank)
print(f"Prepared {N_SPLITS} CV folds with feature dimensions: {FEATURE_DIMS}")


Prepared 5 CV folds with feature dimensions: {'semantic': 768, 'structural': 20, 'phonetic': 44, 'char_svd': 64}


## Global partition model and loss

The shared encoder contextualizes the words; the shared set decoder produces four puzzle-dependent slots. A new MLP scores word/slot compatibility, giving logits of shape `(B, 16, 4)`.

Training minimizes word cross-entropy over all 24 alignments between gold group IDs and predicted slots. This is equivalent to four-by-four Hungarian matching with the same additive loss, and stays on the GPU. Gradients flow through the selected alignment. A 16-by-16 Hungarian assignment at decoding time assigns each word once and gives each slot exactly four words. Training uses soft assignments; decoding enforces the hard capacity constraints.

Each example contains all 16 words from one date. The four feature branches are projected separately and combined by learned per-word gates before the shared set encoder. Word shuffles preserve targets; this model has no positional encodings.

In [6]:
DEFAULT_MODEL_CONFIG = {
    "feature_dims": FEATURE_DIMS, "width": 256, "layers": 2, "heads": 8,
    "scorer_width": 128, "dropout": 0.3,
}
DEFAULT_TRAINING_CONFIG = {"lr": 1e-4, "weight_decay": 1e-4}


## Training and validation selection

Both training and evaluation use the selected device. Checkpoints and Optuna trials are selected by validation solve rate, with mean exact groups as a small tie-breaker. Training stops after 30 consecutive epochs without either a new best solve/group score or a validation CE decrease of at least `EARLY_STOPPING_MIN_DELTA`, up to `EPOCHS=200`. Loss improvements are measured against the last significant best, so small cumulative gains count. Early stopping and checkpoint selection are separate: the selected checkpoint still prioritizes solve rate. The test split is evaluated only after selecting the model. Raw word accuracy is reported after the best permutation of group labels.

In [7]:
def selection_score(metrics):
    # One extra solved puzzle dominates this bounded tie-breaker.
    epsilon = min(0.001, 0.5 / metrics["puzzles"])
    return metrics["solve_rate"] + epsilon * metrics["mean_correct_groups"] / 4

def train_partition_model(model_config, training_config, train_loader, val_loader):
    seed_experiment(SEED)
    model = PartitionTransformer(**model_config).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), **training_config)
    best_key = (-float("inf"), -float("inf"))
    best_state = best_metrics = None
    best_epoch = None
    stopping_loss = float("inf")
    stale_epochs = 0
    history = []
    for epoch in range(EPOCHS):
        model.train()
        loss_sum = examples = 0
        for batch in train_loader:
            x = {name: value.to(device) for name, value in batch["x"].items()}
            targets = batch["targets"].to(device)
            optimizer.zero_grad(set_to_none=True)
            loss = partition_loss(model(x), targets)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            loss_sum += loss.item() * len(targets)
            examples += len(targets)
        metrics = evaluate_partitions(model, val_loader, device)
        score = selection_score(metrics)
        key = (score, -metrics["loss"])
        score_improved = score > best_key[0]
        loss_improved = metrics["loss"] < stopping_loss - EARLY_STOPPING_MIN_DELTA
        if loss_improved:
            stopping_loss = metrics["loss"]
        stale_epochs = 0 if score_improved or loss_improved else stale_epochs + 1
        if key > best_key:
            best_key = key
            best_epoch = epoch
            best_metrics = metrics.copy()
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        history.append({
            "epoch": epoch, "train_loss": loss_sum / examples,
            **{f"val_{k}": v for k, v in metrics.items()},
        })
        if epoch % 5 == 0 or epoch == EPOCHS - 1:
            print(
                f"Epoch {epoch} | Train CE: {loss_sum / examples:.4f} | "
                f"Val CE: {metrics['loss']:.4f} | "
                f"Val solve: {metrics['solve_rate']:.1%} | "
                f"Val exact groups: {metrics['mean_correct_groups']:.2f}/4"
            )
        if stale_epochs >= EARLY_STOPPING_PATIENCE:
            print(f"Early stopping after {epoch + 1} epochs; selected epoch {best_epoch + 1}.")
            break
    return {
        "score": best_key[0], "state": best_state, "metrics": best_metrics,
        "model_config": model_config.copy(), "training_config": training_config.copy(),
        "history": history,
        "epochs_run": len(history), "best_epoch": best_epoch + 1,
    }


## Five-fold Optuna search

Each Optuna trial is scored by the mean validation solve/group score across five date-level folds drawn only from the training split. Fold-specific character and structural transforms are fitted only on each fold's training dates. After selecting hyperparameters, the final model trains on the full training split and uses the separate validation split for early stopping. The test split is reserved for final evaluation. This costs about five times as much per trial. Set `RUN_OPTUNA=False` for a single default-configuration fit.

In [8]:
def objective(trial):
    model_config = {
        "feature_dims": FEATURE_DIMS,
        "width": trial.suggest_categorical("width", [128, 256, 512]),
        "layers": trial.suggest_int("layers", 1, 3),
        "heads": 8,
        "scorer_width": trial.suggest_categorical("scorer_width", [128, 256]),
        "dropout": trial.suggest_float("dropout", 0.1, 0.5),
    }
    training_config = {
        "lr": trial.suggest_float("lr", 1e-5, 3e-4, log=True),
        "weight_decay": trial.suggest_float("weight_decay", 1e-6, 1e-3, log=True),
    }
    fold_scores, fold_metrics = [], []
    for fold_idx, (fold_train_loader, fold_val_loader) in enumerate(cv_loaders):
        run = train_partition_model(
            model_config, training_config, fold_train_loader, fold_val_loader,
        )
        fold_scores.append(run["score"])
        fold_metrics.append(run["metrics"])
        trial.report(float(np.mean(fold_scores)), fold_idx)
    trial.set_user_attr("fold_scores", fold_scores)
    trial.set_user_attr("fold_solve_rates", [m["solve_rate"] for m in fold_metrics])
    trial.set_user_attr("mean_cv_groups", float(np.mean([m["mean_correct_groups"] for m in fold_metrics])))
    return float(np.mean(fold_scores))

if RUN_OPTUNA:
    study = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=SEED),
        pruner=optuna.pruners.NopPruner(),
    )
    study.optimize(objective, n_trials=OPTUNA_TRIALS, gc_after_trial=True)
    best_model_config = {
        "feature_dims": FEATURE_DIMS,
        "width": study.best_params["width"],
        "layers": study.best_params["layers"],
        "heads": 8,
        "scorer_width": study.best_params["scorer_width"],
        "dropout": study.best_params["dropout"],
    }
    best_training_config = {
        "lr": study.best_params["lr"],
        "weight_decay": study.best_params["weight_decay"],
    }
else:
    best_model_config, best_training_config = DEFAULT_MODEL_CONFIG, DEFAULT_TRAINING_CONFIG

best_run = train_partition_model(
    best_model_config, best_training_config, train_loader, val_loader,
)

model = PartitionTransformer(**best_run["model_config"]).to(device)
model.load_state_dict(best_run["state"])
model.eval()
print("Selected configuration:", best_run["model_config"], best_run["training_config"])
print("Selected validation metrics:", best_run["metrics"])


[I 2026-10-09 13:30:29,239] A new study created in memory with name: no-name-faaccd96-a663-4ada-884a-20639cc49cbb


Epoch 0 | Train CE: 1.2906 | Val CE: 1.3634 | Val solve: 0.0% | Val exact groups: 0.01/4
Epoch 5 | Train CE: 1.0841 | Val CE: 1.1667 | Val solve: 0.0% | Val exact groups: 0.06/4
Epoch 10 | Train CE: 0.5659 | Val CE: 1.5101 | Val solve: 0.0% | Val exact groups: 0.03/4
Epoch 15 | Train CE: 0.2147 | Val CE: 2.3615 | Val solve: 0.0% | Val exact groups: 0.05/4
Epoch 20 | Train CE: 0.1066 | Val CE: 2.9982 | Val solve: 0.0% | Val exact groups: 0.06/4
Epoch 25 | Train CE: 0.0835 | Val CE: 3.1300 | Val solve: 0.0% | Val exact groups: 0.08/4
Epoch 30 | Train CE: 0.0616 | Val CE: 3.4974 | Val solve: 0.0% | Val exact groups: 0.06/4
Epoch 35 | Train CE: 0.0558 | Val CE: 3.4103 | Val solve: 0.0% | Val exact groups: 0.06/4
Epoch 40 | Train CE: 0.0441 | Val CE: 3.6239 | Val solve: 0.0% | Val exact groups: 0.09/4
Epoch 45 | Train CE: 0.0357 | Val CE: 3.8931 | Val solve: 0.0% | Val exact groups: 0.07/4
Epoch 50 | Train CE: 0.0345 | Val CE: 4.0352 | Val solve: 0.0% | Val exact groups: 0.08/4
Epoch 55 | T

[I 2026-10-09 13:33:30,040] Trial 0 finished with value: 0.0010571977992628598 and parameters: {'width': 256, 'layers': 2, 'scorer_width': 256, 'dropout': 0.275034884505077, 'lr': 0.00020761410420015315, 'weight_decay': 0.0007780155576901416}. Best is trial 0 with value: 0.0010571977992628598.


Early stopping after 80 epochs; selected epoch 50.
Epoch 0 | Train CE: 1.3504 | Val CE: 1.3553 | Val solve: 0.0% | Val exact groups: 0.02/4
Epoch 5 | Train CE: 1.2630 | Val CE: 1.2692 | Val solve: 0.0% | Val exact groups: 0.01/4
Epoch 10 | Train CE: 1.2292 | Val CE: 1.2428 | Val solve: 0.0% | Val exact groups: 0.02/4
Epoch 15 | Train CE: 1.1863 | Val CE: 1.2214 | Val solve: 0.0% | Val exact groups: 0.03/4
Epoch 20 | Train CE: 1.1426 | Val CE: 1.2085 | Val solve: 0.0% | Val exact groups: 0.04/4
Epoch 25 | Train CE: 1.1030 | Val CE: 1.2028 | Val solve: 0.0% | Val exact groups: 0.03/4
Epoch 30 | Train CE: 1.0693 | Val CE: 1.2039 | Val solve: 0.0% | Val exact groups: 0.04/4
Epoch 35 | Train CE: 1.0335 | Val CE: 1.2089 | Val solve: 0.0% | Val exact groups: 0.05/4
Epoch 40 | Train CE: 1.0001 | Val CE: 1.2157 | Val solve: 0.0% | Val exact groups: 0.04/4
Epoch 45 | Train CE: 0.9665 | Val CE: 1.2259 | Val solve: 0.0% | Val exact groups: 0.04/4
Epoch 50 | Train CE: 0.9318 | Val CE: 1.2387 | Val 

[I 2026-10-09 13:36:37,476] Trial 1 finished with value: 1.2914641311895733e-05 and parameters: {'width': 256, 'layers': 2, 'scorer_width': 128, 'dropout': 0.13485171988061628, 'lr': 1.0711863369746426e-05, 'weight_decay': 0.0003146730406166008}. Best is trial 0 with value: 0.0010571977992628598.


Epoch 80 | Train CE: 0.6969 | Val CE: 1.3771 | Val solve: 0.0% | Val exact groups: 0.05/4
Early stopping after 81 epochs; selected epoch 51.
Epoch 0 | Train CE: 1.2850 | Val CE: 1.2470 | Val solve: 0.0% | Val exact groups: 0.03/4
Epoch 5 | Train CE: 0.8711 | Val CE: 1.2591 | Val solve: 0.0% | Val exact groups: 0.06/4
Epoch 10 | Train CE: 0.1064 | Val CE: 2.4793 | Val solve: 0.0% | Val exact groups: 0.06/4
Epoch 15 | Train CE: 0.0418 | Val CE: 3.2582 | Val solve: 0.0% | Val exact groups: 0.04/4
Epoch 20 | Train CE: 0.0248 | Val CE: 3.7093 | Val solve: 0.0% | Val exact groups: 0.06/4
Epoch 25 | Train CE: 0.0266 | Val CE: 3.9739 | Val solve: 0.0% | Val exact groups: 0.06/4
Epoch 30 | Train CE: 0.0214 | Val CE: 4.1628 | Val solve: 0.0% | Val exact groups: 0.07/4
Epoch 35 | Train CE: 0.0177 | Val CE: 4.2927 | Val solve: 0.0% | Val exact groups: 0.06/4
Epoch 40 | Train CE: 0.0158 | Val CE: 4.2561 | Val solve: 0.0% | Val exact groups: 0.10/4
Epoch 45 | Train CE: 0.0099 | Val CE: 4.5904 | Val 

... (16 outputs left out)

[I 2026-10-09 14:02:40,662] Trial 10 finished with value: 2.479434859248972e-05 and parameters: {'width': 512, 'layers': 2, 'scorer_width': 256, 'dropout': 0.2862333117175267, 'lr': 7.678711197577428e-05, 'weight_decay': 1.1980338102687282e-05}. Best is trial 2 with value: 0.002082511617969125.


Early stopping after 80 epochs; selected epoch 50.
Epoch 0 | Train CE: 1.3132 | Val CE: 1.2965 | Val solve: 0.0% | Val exact groups: 0.01/4
Epoch 5 | Train CE: 1.0155 | Val CE: 1.2232 | Val solve: 0.0% | Val exact groups: 0.05/4
Epoch 10 | Train CE: 0.6547 | Val CE: 1.4762 | Val solve: 0.0% | Val exact groups: 0.04/4
Epoch 15 | Train CE: 0.2499 | Val CE: 2.2401 | Val solve: 0.0% | Val exact groups: 0.05/4


[W 2026-10-09 14:02:54,157] Trial 11 failed with parameters: {'width': 256, 'layers': 2, 'scorer_width': 256, 'dropout': 0.1994446016972601, 'lr': 0.00011506764394278724, 'weight_decay': 1.5233356226471236e-06} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/mnt/code/python_projects/nyt-connections-solver/.venv/lib/python3.11/site-packages/optuna/study/_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "/tmp/ipykernel_58395/263129568.py", line 16, in objective
    run = train_partition_model(
          ^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_58395/354773303.py", line 29, in train_partition_model
    metrics = evaluate_partitions(model, val_loader, device)
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/mnt/code/python_projects/nyt-connections-solver/.venv/lib/python3.11/site-packages/torch/utils/_contextlib.py", line 116, in decorate_context
    re

KeyboardInterrupt: 

In [ ]:
pd.DataFrame(best_run["history"])


## Final evaluation

One partition per puzzle, with no beam search. Exact solve rate requires all four groups to match, ignoring group labels and word order. Mean exact groups and matched word accuracy provide partial-credit diagnostics. Every decoded result has four disjoint groups of four.

In [ ]:
final_metrics = {
    "train": evaluate_partitions(model, make_loader(train_dates, final_feature_bank), device),
    "val": best_run["metrics"],
    "test": evaluate_partitions(model, test_loader, device),
}
pd.DataFrame.from_dict(final_metrics, orient="index")


## Save global partition weights

This checkpoint uses the separate `PartitionTransformer` architecture and `decode_partition` function. It is not a checkpoint for the existing sequential search harness.

In [ ]:
weights_dir = Path("weights")
weights_dir.mkdir(parents=True, exist_ok=True)
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
weights_path = weights_dir / f"partition_transformer_{timestamp}.pt"
checkpoint = {
    "model_class": "PartitionTransformer",
    "task": "balanced_partition",
    "model_config": best_run["model_config"],
    "training_config": {
        **best_run["training_config"], "epochs": best_run["epochs_run"], "seed": SEED,
        "max_epochs": EPOCHS, "best_epoch": best_run["best_epoch"],
        "early_stopping_patience": EARLY_STOPPING_PATIENCE,
        "early_stopping_min_delta": EARLY_STOPPING_MIN_DELTA,
        "loss": "permutation_invariant_cross_entropy",
    },
    "model_state_dict": {k: v.detach().cpu() for k, v in model.state_dict().items()},
    "embedding_model_name": EMBEDDING_MODEL,
    "feature_encoder": {
        "char_vocabulary": final_feature_encoder.vectorizer.vocabulary_,
        "char_svd_components": torch.tensor(final_feature_encoder.svd.components_),
        "structural_mean": torch.tensor(final_feature_encoder.struct_mean),
        "structural_std": torch.tensor(final_feature_encoder.struct_std),
        "feature_dims": FEATURE_DIMS,
    },
    "cv_folds": N_SPLITS,
    "dataset_path": str(DATASET_PATH.resolve()),
    "split_dates": {name: list(map(str, dates)) for name, dates in
                    (("train", train_dates), ("val", val_dates), ("test", test_dates))},
    "validation_metrics": best_run["metrics"],
}
torch.save(checkpoint, weights_path)
print("Saved global partition checkpoint:", weights_path)
